In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import math
import seaborn as sns

In [ ]:
# Read the 100 EDM datasets
files = sorted(glob.glob("../data/bootstrap/SHAP_discrepancy_rs_*.csv"))

all_df = []

for file in files: 
    rs = int(os.path.basename(file).split("_")[-1].replace(".csv",""))

    df = pd.read_csv(file)
    df["RandomSeed"] = rs

    all_df.append(df) # add the dataframe to the created list

all_df = pd.concat(all_df, ignore_index= True)  

print(all_df.shape)
all_df

In [ ]:
# In the original data sets of SHAP_discrepancy, 'Site' column represents the site and year index and Mean_over_SDofMean = EDM. 
# Rename them to make them easier to understand.
all_df.rename(columns ={
    'Site': 'Site_yr_idx',
    'Mean_over_SDofMean': 'EDM'
}, inplace = True)

In [ ]:
# Plot the boxplot of overall EDM (SHAP discrepancy) 
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    "legend.fontsize": 12,
    "legend.title_fontsize": 14,
    "legend.frameon": False,
})


model_order = ['GAM','RF', 'BRT', 'MLP']
ref_palette = {
    'GAM': sns.color_palette("Purples", 4)[1],
    'RF':  sns.color_palette("Blues", 4)[1],
    'BRT': sns.color_palette("Greens", 4)[1],
    'MLP': sns.color_palette("Reds", 4)[1]
}


plt.figure(figsize=(8, 6))
sns.boxplot(
    data=all_df,
    x='Reference_Model',
    y='EDM',
    hue='Reference_Model',
    order=model_order,
    palette=ref_palette,
    showfliers=False
)

plt.xlabel("Reference Model", fontsize=16,labelpad=10)
plt.ylabel("Overall EDM (SHAP discrepancy)", fontsize=16, labelpad=10)
plt.tick_params(axis = 'x', labelsize =14)
plt.tick_params(axis = 'y', labelsize =14)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Calculate the 95% bootstrap interval.

summary =  (
    all_df.groupby(["Site_yr_idx", "Reference_Model"]).agg(
        Mean_EDM = ("EDM", "mean"),
        SD_EDM = ("EDM", "std"),
        N = ("Site_yr_idx", "count"),
        lower95 = ("EDM", lambda x: x.quantile(0.025)),
        upper95 = ("EDM", lambda x: x.quantile(0.975))      
    ).reset_index()
)

In [ ]:
# Calculate the standard error
summary["SE"] = summary["SD_EDM"]/np.sqrt(summary["N"])

# Calculate 95% CI
summary["95% LB"] = summary["Mean_EDM"] - 1.96 * summary["SE"]
summary["95% UB"] = summary["Mean_EDM"] + 1.96 * summary["SE"]

In [ ]:
# Plot the bootstrap intervals for E1, E2, and E3 across reference models.

fig, ax = plt.subplots(figsize=(7,7))

sns.boxplot(
    data=all_df,
    x='Reference_Model',
    y='EDM',
    hue='Reference_Model',
    order=model_order,
    palette=ref_palette,
    showfliers=False
)


important_sites = [
    1419,   # E1
    158,    # E2
    200     # E3
]

labels = {
    1419: r"$E_1$",
    158: r"$E_2$",
    200: r"$E_3$"
}

model_positions = {
    "GAM":0,
    "RF":1,
    "BRT":2,
    "MLP":3
}

# horizontal offsets
offsets = [-0.12,0,0.20]

for i, site in enumerate(important_sites):

    df_site = summary[summary["Site_yr_idx"] == site]

    for _, row in df_site.iterrows():

        x = model_positions[row["Reference_Model"]] + offsets[i]

        mean = row["Mean_EDM"]

        lower = row["lower95"]
        upper = row["upper95"]

        ax.errorbar(
            x,
            mean,
            yerr=[[mean-lower],[upper-mean]],
            fmt='o',
            color='red',
            markersize=5,
            capsize=5,
            linewidth=2,
            zorder=10
        )

        ax.text(
            x+0.04,
            mean,
            labels[site],
            fontsize=10,
            color="red"
        )

ax.set_xlabel("Reference Model")
ax.set_ylabel("Overall EDM")
plt.tight_layout()
plt.show()

In [ ]:
# Calculate the 95% bootstrap interval ignoring reference model
summary2 =  (
    all_df.groupby(["Site_yr_idx"]).agg(
        Mean_EDM = ("EDM", "mean"),
        SD_EDM = ("EDM", "std"),
        N = ("Site_yr_idx", "count"),
        lower95 = ("EDM", lambda x: x.quantile(0.025)),
        upper95 = ("EDM", lambda x: x.quantile(0.975))      
    ).reset_index()
)

In [ ]:
# Calculate the standard error
summary2["SE"] = summary2["SD_EDM"]/np.sqrt(summary2["N"])

# Caalculate 95% CI
summary2["95% LB"] = summary2["Mean_EDM"] - 1.96 * summary2["SE"]
summary2["95% UB"] = summary2["Mean_EDM"] + 1.96 * summary2["SE"]

In [ ]:
# Plot the bootstrap intervals for E1, E2, and E3.
fig, ax = plt.subplots(figsize=(3, 7))

# One overall boxplot
sns.boxplot(
    y=all_df["EDM"],
    color="lightgrey",
    showfliers=False,
    width=0.4,
    ax=ax
)

important_sites = [1419, 158, 200]

labels = {
    1419: r"$E_1$",
    158:  r"$E_2$",
    200:  r"$E_3$"
}

offsets = [-0.08, 0.0, 0.08]

for i, site in enumerate(important_sites):

    df_site = summary2[summary2["Site_yr_idx"] == site]

    # Overall mean and CI across all reference models
    mean = df_site["Mean_EDM"].item()
    lower = df_site["lower95"].item()
    upper = df_site["upper95"].item()

    x = offsets[i]

    ax.errorbar(
        x,
        mean,
        yerr=[[mean - lower], [upper - mean]],
        fmt='o',
        color='red',
        markersize=6,
        capsize=5,
        linewidth=2,
        zorder=10
    )

    ax.text(
        x + 0.02,
        mean,
        labels[site],
        color="red",
        fontsize=11
    )

ax.set_xticks([])
ax.set_xlabel("")
ax.set_ylabel("Overall EDM")

plt.tight_layout()
plt.show()

In [ ]:
# Plot the bootstrap intervals horizontally for E1, E2, and E3.
fig, ax = plt.subplots(figsize=(7, 3))

sns.boxplot(
    x=all_df["EDM"],
    color="lightgrey",
    showfliers=False,
    width=0.4,
    ax=ax
)

important_sites = [1419, 158, 200]

labels = {
    1419: r"$E_1$",
    158:  r"$E_2$",
    200:  r"$E_3$"
}

offsets = [-0.08, 0.0, 0.08]

for i, site in enumerate(important_sites):

    df_site = summary2[summary2["Site_yr_idx"] == site]

    # Overall mean and CI across all reference models
    mean = df_site["Mean_EDM"].item()
    lower = df_site["lower95"].item()
    upper = df_site["upper95"].item()

    y = offsets[i]

    ax.errorbar(
        mean,
        y,
        xerr=[[mean - lower], [upper - mean]],
        fmt='o',
        color='red',
        markersize=6,
        capsize=5,
        linewidth=2,
        zorder=10
    )

    ax.text(
        mean,
        y + 0.05,
        labels[site],
        color="red",
        fontsize=11
    )

ax.set_yticks([])
ax.set_ylabel("")
ax.set_xlabel("Overall EDM")

plt.tight_layout()
plt.show()